# Entity Extraction Pipeline

This notebook implements the core "AI" of the project. We use **spaCy** combined with a Rule-Based **EntityRuler** to extract medical entities from the text.

**Entities to Extract:**
*   **PROBLEM**: Diseases, symptoms, disorders (e.g., "pain", "fracture", "pneumonia").
*   **TREATMENT**: Medications, surgeries, procedures (e.g., "aspirin", "incision", "therapy").
*   **TEST**: Diagnostic tests (e.g., "x-ray", "scan", "labs").

In [1]:
import pandas as pd
import spacy
from spacy.pipeline import EntityRuler
from spacy import displacy
import os

# Settings
pd.set_option('display.max_colwidth', 150)

# Paths
DATA_PATH = "../data/mtsamples.csv"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Setup SpaCy Pipeline with EntityRuler
We use the small English model and add specific rules for medical terms.

In [2]:
nlp = spacy.load("en_core_web_sm")

# Create the EntityRuler
if "entity_ruler" not in nlp.pipe_names:
    ruler = nlp.add_pipe("entity_ruler", before="ner")
else:
    ruler = nlp.get_pipe("entity_ruler")

# Define Patterns (A small sample of common terms to demonstrate the concept)
patterns = [
    # PROBLEMS
    {"label": "PROBLEM", "pattern": [{"LOWER": "pain"}]},
    {"label": "PROBLEM", "pattern": [{"LOWER": "fracture"}]},
    {"label": "PROBLEM", "pattern": [{"LOWER": "disease"}]},
    {"label": "PROBLEM", "pattern": [{"LOWER": "disorder"}]},
    {"label": "PROBLEM", "pattern": [{"LOWER": "syndrome"}]},
    {"label": "PROBLEM", "pattern": [{"LOWER": "pneumonia"}]},
    {"label": "PROBLEM", "pattern": [{"LOWER": "fibrillation"}]},
    {"label": "PROBLEM", "pattern": [{"LOWER": "hypertension"}]},
    {"label": "PROBLEM", "pattern": [{"LOWER": "diabetes"}]},
    
    # TREATMENTS
    {"label": "TREATMENT", "pattern": [{"LOWER": "surgery"}]},
    {"label": "TREATMENT", "pattern": [{"LOWER": "incision"}]},
    {"label": "TREATMENT", "pattern": [{"LOWER": "excision"}]},
    {"label": "TREATMENT", "pattern": [{"LOWER": "therapy"}]},
    {"label": "TREATMENT", "pattern": [{"LOWER": "medication"}]},
    {"label": "TREATMENT", "pattern": [{"LOWER": "aspirin"}]},
    {"label": "TREATMENT", "pattern": [{"LOWER": "mg"}]},  # often indicates medication dosage
    {"label": "TREATMENT", "pattern": [{"LOWER": "tablet"}]},
    
    # TESTS
    {"label": "TEST", "pattern": [{"LOWER": "x-ray"}]},
    {"label": "TEST", "pattern": [{"LOWER": "scan"}]},
    {"label": "TEST", "pattern": [{"LOWER": "mri"}]},
    {"label": "TEST", "pattern": [{"LOWER": "ct"}]},
    {"label": "TEST", "pattern": [{"LOWER": "ultrasound"}]},
    {"label": "TEST", "pattern": [{"LOWER": "exam"}]},
    {"label": "TEST", "pattern": [{"LOWER": "labs"}]}
]

ruler.add_patterns(patterns)
print("Pipeline setup complete. Pipe names:", nlp.pipe_names)

Pipeline setup complete. Pipe names: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'ner']


## 2. Load and Prepare Data

In [3]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['transcription'])
# Use a subset for faster processing in this demo, or process all if feasible
df_sample = df.head(100).copy() 
print(f"Processing {len(df_sample)} notes...")

Processing 100 notes...


## 3. Extraction Function

In [4]:
def extract_entities(text):
    doc = nlp(text)
    entities = {
        "PROBLEM": [],
        "TREATMENT": [],
        "TEST": []
    }
    for ent in doc.ents:
        if ent.label_ in entities:
            entities[ent.label_].append(ent.text)
            
    # Remove duplicates
    for k in entities:
        entities[k] = list(set(entities[k]))
        
    return entities

# Apply to dataframe
extracted_data = df_sample['transcription'].apply(extract_entities)
df_sample['problems'] = extracted_data.apply(lambda x: x['PROBLEM'])
df_sample['treatments'] = extracted_data.apply(lambda x: x['TREATMENT'])
df_sample['tests'] = extracted_data.apply(lambda x: x['TEST'])

print("Extraction complete.")

Extraction complete.


## 4. Visualizing Results

In [5]:
# Pick a note with some findings
sample_idx = df_sample[df_sample['problems'].apply(len) > 0].index[0]
text = df_sample.loc[sample_idx, 'transcription']

print(f"Sample Note ID: {sample_idx}")
doc = nlp(text)
colors = {"PROBLEM": "#ff9999", "TREATMENT": "#99ff99", "TEST": "#9999ff"}
options = {"ents": ["PROBLEM", "TREATMENT", "TEST"], "colors": colors}

displacy.render(doc, style="ent", options=options, jupyter=True)

Sample Note ID: 1


## 5. Save Processed Data

In [6]:
output_file = os.path.join(OUTPUT_DIR, "tagged_notes.csv")
cols_to_save = ['medical_specialty', 'transcription', 'problems', 'treatments', 'tests']
df_sample[cols_to_save].to_csv(output_file, index=False)
print(f"Saved tagged notes to {output_file}")
df_sample[cols_to_save].head()

Saved tagged notes to ../data/processed\tagged_notes.csv


,medical_specialty,transcription,problems,treatments,tests
0,Allergy / Immunology,"SUBJECTIVE:, This 23-year-old white female presents with complaint of allergies. She used to have allergies when she lived in Seattle but she th...",[],[medication],[]
1,Bariatrics,"PAST MEDICAL HISTORY:, He has difficulty climbing stairs, difficulty with airline seats, tying shoes, used to public seating, and lifting objects ...","[hypertension, diabetes, disease, fibrillation, pain]",[surgery],[]
2,Bariatrics,"HISTORY OF PRESENT ILLNESS: , I have seen ABC today. He is a very pleasant gentleman who is 42 years old, 344 pounds. He is 5'9"". He has a BMI ...","[hypertension, diabetes, disease, fibrillation, pain]","[medication, surgery]",[]
3,Cardiovascular / Pulmonary,"2-D M-MODE: , ,1. Left atrial enlargement with left atrial diameter of 4.7 cm.,2. Normal size right and left ventricle.,3. Normal LV systolic f...",[],[],[]
4,Cardiovascular / Pulmonary,1. The left ventricular cavity size and wall thickness appear normal. The wall motion and left ventricular systolic function appears hyperdynami...,[hypertension],[],[]
